In [ ]:
import numpy as np
import glob
import os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from skimage.io import imread
from skimage.transform import resize

# 1. Load images and ages from UTKFace dataset
data_path = '/Users/rongrongqian/Desktop/AI modules/regression/UTKface/part1/*.jpg' #change the path
image_files = glob.glob(data_path)

ages = []
images_resized = []
img_size = (128, 128)  # All images resized to this shape

# 2. Read, resize, and parse age from filenames
for file in image_files[:100]:  # For demo, use first 100 images
    age = int(os.path.basename(file).split('_')[0])  # Filename format: age_...
    img = imread(file)
    img_resized = resize(img, img_size, anti_aliasing=True)
    ages.append(age)
    images_resized.append(img_resized)

ages = np.array(ages)
images_resized = np.array(images_resized)

# 3. Split into training and test sets
X_train_img , X_test_img , y_train , y_test = train_test_split (
    images_resized , ages , test_size =0.2 , random_state =42
)

# 4. Flatten images for PCA input 
X_train = X_train_img.reshape(X_train_img.shape[0], -1)
X_test = X_test_img.reshape(X_test_img.shape[0], -1)

# 5. Standardize features (important before PCA)
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

# 6. Fit PCA on training set and apply to both train/test
pca = PCA(n_components=60, random_state=42)
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)

# 7. OLS regression
pca_model_ols = LinearRegression().fit(X_train_pca, y_train)
pca_pred_ols = pca_model_ols.predict(X_test_pca)
mae_ols = mean_absolute_error(y_test, pca_pred_ols)
print("PCA Features OLS MAE:", mae_ols)

# 8. Ridge regression with cross-validation
pca_model_ridgecv = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], cv=5)
pca_model_ridgecv.fit(X_train_pca, y_train)
pca_pred_ridgecv = pca_model_ridgecv.predict(X_test_pca)
mae_ridgecv = mean_absolute_error(y_test, pca_pred_ridgecv)
print("PCA Features RidgeCV Best Alpha:", pca_model_ridgecv.alpha_)
print("PCA Features RidgeCV MAE:", mae_ridgecv)

# 9. Lasso regression with cross-validation
pca_model_lassocv = LassoCV(alphas=[0.001, 0.01, 0.1, 1, 10], cv=5, max_iter=5000)
pca_model_lassocv.fit(X_train_pca, y_train)
pca_pred_lassocv = pca_model_lassocv.predict(X_test_pca)
mae_lassocv = mean_absolute_error(y_test, pca_pred_lassocv)
print("PCA Features LassoCV Best Alpha:", pca_model_lassocv.alpha_)
print("PCA Features LassoCV MAE:", mae_lassocv)


PCA Features OLS MAE: 19.56249704347364
PCA Features RidgeCV Best Alpha: 100.0
PCA Features RidgeCV MAE: 19.547862433486536
PCA Features LassoCV Best Alpha: 10.0
PCA Features LassoCV MAE: 18.791380663943233
